# Chapter 7 — A gap you can prove

Two things in this course have been *forgetting*. In chapter 4, the heat-bath sampler started from an arbitrary labeling and, after enough sweeps, forgot where it started. In chapter 5, two layers of a board forgot each other at a rate we called the gap. We measured both, but proved neither.

This chapter proves that they are the same phenomenon, and turns it into a theorem. If every edge listens only weakly to its neighbors, then two samplers fed the same random numbers are forced to agree. From that one fact follow decaying correlations and then a positive gap, with an explicit number. The result holds for every compact label group, every dimension and every board width. It is the small-stiffness end of the mass-gap story of chapter 6 (physicists say "strong coupling"), proved from scratch. Its method is a standard tool of computer science, the *coupling* argument used to show that Markov chain samplers run fast.

The last section is the honest part. It explains exactly why this theorem, and every theorem of its kind, stops short of the Millennium problem. It also explains what the 2026 existence proof does and does not supply toward closing the distance.

**What you'll learn**

- How much one edge "listens" to another (its **influence**), and the one-number test $\alpha<1$.
- How running two samplers on the same random numbers proves that they forget their starting points. This is the coupling method of computer science.
- How forgetting gives decaying correlations and then a gap $E_1\ge\ln(1/\alpha)$, and why no argument of this kind reaches the continuum.

**Prerequisites:** chapters 2 (models), 4 (heat bath, face remainders) and 5 (transfer matrix, levels). The last section uses chapter 6.

In [ ]:
# Running in the browser: install the course library first.
%pip install -q lgt

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import LinearOperator, eigsh

from lgt import *
from lgt import viz
from lgt.grid import GridModel

viz.use_style()
rng = np.random.default_rng(7)

## 7.1 How much does one edge listen to another?

Recall the heat bath (chapter 4). To update one edge $e$, you hold every other label fixed and draw $U(e)$ from its **conditional distribution**. That distribution depends only on the face remainders of the faces through $e$ (glossary 4.8):

$$
\mu_e(dx\mid U)\ \propto\ \exp\Big(-\beta\sum_{f\ni e}\varphi\big(x\,r_f(e)\big)\Big)\,\sigma(dx).
$$

Change the label of some other edge $e'$, and the conditional distribution at $e$ may move. If $e$ and $e'$ share no face, it does not move at all. If they share a face, it moves a little. To say "a little" precisely, we need a distance between distributions.

**Definition (distance between distributions).** For two distributions $p,q$ on the same set, $d(p,q)\in[0,1]$ is the largest difference in probability they assign to any event:

$$
d(p,q)=\max_{A}\,\big|p(A)-q(A)\big|=\tfrac12\sum_x|p(x)-q(x)| .
$$

For distributions with densities, the sum becomes an integral. *(Probabilists call this the total variation distance.)*

**Definition (influence).** For edges $e\neq e'$, the influence of $e'$ on $e$ is $C(e,e')\in[0,1]$:

$$
C(e,e')=\max\Big\lbrace d\big(\mu_e(\cdot\mid U),\,\mu_e(\cdot\mid U')\big)\ :\ U,U'\text{ differ only at }e'\Big\rbrace .
$$

Set $C(e,e)=0$. The **influence number** of the model is the most any edge listens in total:

$$
\alpha=\max_e\sum_{e'}C(e,e').
$$

Finally, the **influence distance** between two sets of edges is the fewest steps between them, where one step moves between two edges that share a face. *(Probabilists call $C$ Dobrushin's interdependence matrix, and $\alpha<1$ Dobrushin's condition.)*

The influence of one face on an edge is controlled by a small lemma about tilting a distribution. Here $\operatorname{tanh}$ is the hyperbolic tangent; for small $c$, $\tanh(c/4)\approx c/4$.

> **Fact 7.1 (tilting by a bounded factor).** Let $p$ be a distribution and $g$ a positive function whose largest value is at most $e^{c}$ times its smallest. Let $p_g\propto g\,p$ be $p$ reweighted by $g$. Then $d(p,p_g)\le\tanh(c/4)$, with equality for a well-chosen two-point $p$.

*Proof.* Rescale $g$ so that it takes values in $[b,1]$ with $b=e^{-c}$, and let $m=\langle g\rangle_p\in[b,1]$. Then

$$
d(p,p_g)=\big\langle\big(1-g/m\big)_+\big\rangle_p .
$$

Fix $m$. The function $g\mapsto(1-g/m)_+$ is convex, so it lies below its chord between $b$ and $1$. At $g=1$ it is $0$, because $m\le1$. Averaging the chord gives $d\le s\,(1-b/m)$, where $s$ is the weight the chord puts on $b$; the mean constraint forces $s=(1-m)/(1-b)$. Substituting,

$$
d\le\frac{(1-m)(m-b)}{(1-b)\,m}=\frac{1+b-(m+b/m)}{1-b}\le\frac{1+b-2\sqrt b}{1-b}=\frac{1-\sqrt b}{1+\sqrt b}=\tanh\frac c4,
$$

using $m+b/m\ge2\sqrt b$. Equality holds when $g$ takes only the values $b$ and $1$ and $m=\sqrt b$. $\square$

Now apply it to the grid. Let $\Phi=\max\varphi-\min\varphi$ be the **spread** of the face cost. For the coin model, $\Phi=2$. For $SU(2)$ with $\varphi=2-\operatorname{tr}$, $\Phi=4$.

> **Fact 7.2 (influence on a grid).** On a $d$-dimensional torus grid with every side at least 3, take any compact $G$ and any face cost of spread $\Phi$. Then $C(e,e')\le\tanh(\beta\Phi/2)$ when $e$ and $e'$ share a face, and $C(e,e')=0$ otherwise. Every edge shares a face with exactly $6(d-1)$ other edges. So the influence number satisfies $\alpha\le6(d-1)\tanh(\beta\Phi/2)$.

*Proof.* With side at least 3, two edges share at most one face $f$. Changing $U(e')$ changes only $r_f(e)$, so it reweights $\mu_e$ by $g(x)=\exp\big(-\beta[\varphi(xr_f')-\varphi(xr_f)]\big)$. The exponent lies in $[-\beta\Phi,\beta\Phi]$, so $\max g/\min g\le e^{2\beta\Phi}$. Fact 7.1 with $c=2\beta\Phi$ gives the bound. For the count: $e$ lies in $2(d-1)$ faces, each with 3 other edges, all distinct. $\square$

Let's check Fact 7.2 against the actual conditional distributions. For the coin model we can try every configuration of the remainders. For $SU(2)$ we draw random remainders, including the worst kind of change (one remainder flipped to its negative), and compute the distance by sampling the 3-sphere.

In [ ]:
def coin_influence(beta, d):
    """Exact worst influence for the coin model: P(+1) = logistic(2 beta * sum of remainders)."""
    worst = 0.0
    for rs in itertools.product([1, -1], repeat=2 * (d - 1)):
        A, A2 = sum(rs), sum(rs) - 2 * rs[0]           # change the first face's remainder
        p, p2 = 1 / (1 + np.exp(-2 * beta * A)), 1 / (1 + np.exp(-2 * beta * A2))
        worst = max(worst, abs(p - p2))
    return worst

def su2_influence(beta, d, trials=60, n=100_000):
    """Sampled worst influence for SU(2): density on S^3 proportional to exp(2 beta <x, sum of remainders>)."""
    G, worst = SU2(), 0.0
    x = G.uniform(rng, (n,))
    for _ in range(trials):
        r = G.uniform(rng, (2 * (d - 1),)); r2 = r.copy(); r2[0] = -r[0]
        w1, w2 = np.exp(2 * beta * x @ r.sum(0)), np.exp(2 * beta * x @ r2.sum(0))
        worst = max(worst, 0.5 * np.mean(np.abs(w1 / w1.mean() - w2 / w2.mean())))
    return worst

In [ ]:
print("        beta   worst influence   Fact 7.2 bound")
for beta in [0.02, 0.1, 0.3]:
    print(f"coin  {beta:5.2f}       {coin_influence(beta, 4):.4f}          {np.tanh(beta):.4f}")
for beta in [0.02, 0.1, 0.3]:
    print(f"SU(2) {beta:5.2f}       {su2_influence(beta, 4):.4f}          {np.tanh(2 * beta):.4f}")

Every measured influence sits below its bound. For the coin model at small $\beta$ the bound is almost exact. For $SU(2)$ it is loose by about a factor of 2, because a smooth density on the 3-sphere cannot concentrate the way the worst two-point distribution of Fact 7.1 does.

> **Recap.** The influence $C(e,e')$ is how far the heat-bath distribution at $e$ can move when you change the label at $e'$. On a grid it is at most $\tanh(\beta\Phi/2)$ for neighbors and zero otherwise. So the influence number obeys $\alpha\le6(d-1)\tanh(\beta\Phi/2)$, which is below 1 for small enough $\beta$.

**Check yourself**

1. For the 4D coin model, below which $\beta$ does Fact 7.2 guarantee $\alpha<1$?

<details><summary>Answer</summary>

$18\tanh\beta<1$, so $\beta<\operatorname{artanh}(1/18)\approx0.0556$. Compare the jump that chapter 4 found at $\beta\approx0.44$: the guarantee covers only a small part of the region below the jump.

</details>

2. Why does the influence not depend on how many faces pass through $e$?

<details><summary>Answer</summary>

Changing $U(e')$ changes the remainder of only the one face that $e$ and $e'$ share. The other faces through $e$ reweight both conditional distributions by the same factor, which cancels in the normalization.

</details>

3. What is $\Phi$ for $SU(3)$ with $\varphi=3-\operatorname{Re}\operatorname{tr}$?

<details><summary>Answer</summary>

$\operatorname{Re}\operatorname{tr}$ is a sum of three cosines of angles adding up to a multiple of $2\pi$. It is largest, $3$, at the identity and smallest, $-3/2$, when all three angles are $2\pi/3$, i.e. at $e^{2\pi i/3}I$. So $\Phi=3-(-3/2)=4.5$.

</details>

## 7.2 Two samplers, one set of dice

Here is the idea that powers the rest of the chapter. Run two heat-bath samplers side by side, started from different labelings, and feed them the same random numbers. Whenever their conditional distributions at an edge are close, the shared randomness makes them pick the same label. Disagreements can then only survive by being passed from neighbor to neighbor. If every edge listens to its neighbors with total strength $\alpha<1$, disagreements die faster than they spread.

**Definition (coupling).** A **coupling** of two distributions $p,q$ is a pair of random variables $(X,Y)$, drawn jointly, with $X\sim p$ and $Y\sim q$. For any coupling, $P(X\neq Y)\ge d(p,q)$. There is always a coupling with $P(X\neq Y)=d(p,q)$, called a **best coupling**:

- with probability $1-d$, draw one value from the overlap $\min(p,q)$ (normalized) and set $X=Y$ to it;
- otherwise draw $X$ and $Y$ from the two leftover parts.

For the coin model the heat bath draws $+1$ when a uniform random number $u$ falls below $P(+1)$. Two samplers using the same $u$ disagree exactly when $u$ falls between their two values of $P(+1)$, which happens with probability $|P(+1)-P'(+1)|=d$. So "same random numbers" *is* the best coupling.

> **Fact 7.3 (coupled samplers forget).** Run two heat-bath samplers of the same model, updating edges in the same order, and use a best coupling at every update. Let $\rho_t(e)$ be the probability that the two labels at $e$ differ after $t$ sweeps. Then $\max_e\rho_t(e)\le\alpha^t\,\max_e\rho_0(e)$.

*Proof.* Look at one update of edge $e$, with current labelings $X$ and $Y$. Walk from $X$ to $Y$ by changing their differing labels one at a time. Each change of a label at $e'$ moves the conditional distribution at $e$ by at most $C(e,e')$, so

$$
d\big(\mu_e(\cdot\mid X),\mu_e(\cdot\mid Y)\big)\ \le\ \sum_{e'}C(e,e')\,\mathbf 1[X(e')\neq Y(e')].
$$

The best coupling makes the new labels differ with exactly that probability. Averaging over $X,Y$, the new $\rho(e)$ is at most $\sum_{e'}C(e,e')\rho(e')\le\alpha\max\rho$. So the maximum never grows during a sweep. After a full sweep every edge has been updated once, so every $\rho(e)$ is at most $\alpha$ times the maximum at the start of the sweep. $\square$

Fact 7.3 is the first appearance of the computer-science bridge. If one sampler starts from the model itself, it stays distributed as the model, so the other sampler's labels agree with a true sample except with probability at most $\alpha^t$. With $|E|$ edges, after $t\approx\log|E|/\log(1/\alpha)$ sweeps every label agrees. *(Computer scientists call this the path-coupling proof of rapid mixing. Its best-known use shows that the heat bath for graph colorings mixes fast when there are more than $2\Delta$ colors, $\Delta$ the maximum degree. The structure is the same: bounded degree, weak dependence.)*

Let's watch it happen on the 4D coin model. One sampler starts with every label $+1$, the other with random labels; both use the same random numbers.

In [ ]:
def coupled_disagreement(beta, L=6, sweeps=30, seeds=4):
    """Fraction of edges where two coupled heat-bath samplers disagree, after each sweep."""
    runs = []
    for seed in range(seeds):
        X = GridModel((L,) * 4, Z2(), beta).cold_start()
        Y = GridModel((L,) * 4, Z2(), beta).hot_start(np.random.default_rng(100 + seed))
        rX, rY = np.random.default_rng(seed), np.random.default_rng(seed)   # same dice
        frac = [np.mean(X.U != Y.U)]
        for _ in range(sweeps):
            X.sweep(rX); Y.sweep(rY)
            frac.append(np.mean(X.U != Y.U))
        runs.append(frac)
    return np.mean(runs, axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
t = np.arange(31)
for beta, col in [(0.03, viz.TEAL), (0.05, viz.INK), (0.2, viz.GOLD), (0.6, viz.CORAL)]:
    alpha = 18 * np.tanh(beta)
    ax.semilogy(t, np.maximum(coupled_disagreement(beta), 1e-5), "o-", ms=3, color=col,
                label=rf"$\beta={beta}$, bound on $\alpha$: {alpha:.2f}")
    if alpha < 1:
        ax.semilogy(t, 0.5 * alpha ** t, "--", color=col, lw=1)
ax.set(ylim=(1e-5, 1), xlabel="sweeps", ylabel="fraction of edges that disagree",
       title="4D coin model: two samplers, same random numbers (dashed: Fact 7.3)")
ax.legend(fontsize=9); plt.show()

Look at the two regimes. For $\alpha<1$ (teal, ink), disagreements vanish entirely within a few sweeps, below the dashed guarantee. (The plot floors exact zero at $10^{-5}$.) For $\beta=0.2$ and $0.6$, Fact 7.2 gives no guarantee, and the disagreements never die. They fail in two different ways:

- At $\beta=0.2$, below the jump, the two samplers disagree on about 40% of their *face products* too. Each sampler still forgets its start (chapter 4 checked this by other means). But this particular coupling no longer forces them together. A failed certificate is not proof of slow forgetting.
- At $\beta=0.6$, past the jump, the samplers end up agreeing on about 99% of face products while disagreeing on half the labels. They have reached nearly the same *class* (chapter 1) by different labelings. A coupling that compares labels cannot see that two reframed labelings are "the same".

> **Recap.** A coupling runs two random processes on shared randomness. With the best coupling at every update, the chance that two heat-bath samplers disagree at any edge shrinks by a factor $\alpha$ per sweep. When $\alpha<1$ the sampler forgets its start geometrically fast.

**Check yourself**

1. In the proof, why is it important that the samplers use the *best* coupling rather than independent random numbers?

<details><summary>Answer</summary>

With independent draws, two identical conditional distributions would still give different labels most of the time, so disagreements would never die. The best coupling makes the disagreement probability equal to the distance between the conditional distributions, which is what the influences bound.

</details>

2. On the $6^4$ coin model at $\beta=0.03$, roughly how many sweeps does Fact 7.3 need before all $5184$ labels agree with probability at least $0.99$?

<details><summary>Answer</summary>

$\alpha\le18\tanh0.03\approx0.54$. We need $5184\cdot\tfrac12\alpha^t<0.01$, so $t>\ln(259200)/\ln(1/0.54)\approx20$ sweeps. In the plot it took fewer: the bound is a guarantee, not a prediction.

</details>

## 7.3 From forgetting to decorrelation

Forgetting in *time* (sweeps of a sampler) now gives forgetting in *space* (distance on the board). The trick: a correlation between $A$ and $B$ measures how much conditioning on $B$ moves the average of $A$. Conditioning on $B$ is the same as reweighting the model by $B$. A reweighted model is a new model whose conditional distributions agree with the old one at every edge that $B$ does not read. So we can couple a sampler of the old model with a sampler of the reweighted one. Their disagreements are created only on $B$'s edges, and they fade with influence distance.

For a function $A$ of a labeling, write $\operatorname{osc}(A)=\max A-\min A$. Say $A$ **reads** a set of edges $S_A$ if it depends only on their labels.

> **Fact 7.4 (decorrelation).** Suppose $\alpha<1$. Let $A$ read $S_A$, let $B$ read $S_B$, and let $D$ be the influence distance between $S_A$ and $S_B$. Then the correlation of $A$ and $B$ is at most $\operatorname{osc}(A)\,\operatorname{osc}(B)\,|S_A|\,\alpha^{D}/(1-\alpha)$ in absolute value.

In one line:

$$
\big|\langle AB\rangle-\langle A\rangle\langle B\rangle\big|\ \le\ \operatorname{osc}(A)\,\operatorname{osc}(B)\,|S_A|\,\frac{\alpha^{D}}{1-\alpha}.
$$

*Proof.*

**Step 1 (reweight).** Replace $B$ by $B'=B-\min B\ge0$; this does not change the correlation. If $\langle B'\rangle=0$ the correlation is $0$. Otherwise let $\nu$ be the model reweighted by $B'$, that is, $\nu(dU)=B'(U)\mu(dU)/\langle B'\rangle$. Then $\langle AB\rangle-\langle A\rangle\langle B\rangle=\langle B'\rangle\big(\langle A\rangle_\nu-\langle A\rangle\big)$, and $\langle B'\rangle\le\operatorname{osc}(B)$.

**Step 2 (same conditionals off $S_B$).** At an edge $e\notin S_B$, the factor $B'$ does not depend on $U(e)$. It cancels from the conditional distribution, so $\nu$ and $\mu$ have the same heat bath at $e$.

**Step 3 (couple).** Run a heat-bath sampler $X$ of $\mu$ and one $Y$ of $\nu$, started from samples of $\mu$ and $\nu$, so that at every time $X\sim\mu$ and $Y\sim\nu$. Use best couplings. An update at $e$ produces a disagreement with probability at most $\sum_{e'}C(e,e')\rho(e')$ if $e\notin S_B$, and at most $1$ if $e\in S_B$. Let

$$
\rho^*=\sum_{k\ge0}C^k\,\mathbf 1_{S_B},\qquad\text{so that}\qquad\rho^*=\mathbf 1_{S_B}+C\rho^* .
$$

The sum converges because every row of $C$ sums to at most $\alpha<1$. Exactly as in Fact 7.3, the excess $\max_e(\rho_t(e)-\rho^*(e))_+$ shrinks by a factor $\alpha$ per sweep. So $\rho_t\le\rho^*+\alpha^t$.

**Step 4 (read off $A$).** $\big|\langle A\rangle_\nu-\langle A\rangle\big|=\big|\mathbb E[A(Y_t)-A(X_t)]\big|\le\operatorname{osc}(A)\sum_{e\in S_A}\rho_t(e)$. Now let $t\to\infty$.

**Step 5 (count paths).** $C^k(e,e')$ is a sum over chains of $k$ face-sharing steps from $e$ to $e'$, so it vanishes unless $k\ge D$. Each power satisfies $\sum_{e'}C^k(e,e')\le\alpha^k$. So $\rho^*(e)\le\sum_{k\ge D}\alpha^k=\alpha^D/(1-\alpha)$ for $e\in S_A$. $\square$

Notice what the proof never used: reframings, the group structure, or the dimension. Only the **sparsity** of who-listens-to-whom (a bounded-degree influence graph) and the **weakness** of each link matter. *(Probabilists call this Dobrushin's uniqueness theorem. Computer scientists know its relatives as "spatial mixing" and "correlation decay", the engine of several deterministic approximate-counting algorithms.)*

**Checking Fact 7.4 exactly.** We need a model where correlations at long distances can be computed exactly, not sampled. The transfer matrix of chapter 5 does that. Take the 3D coin model, layered so that each layer is a $3\times3$ torus. Its 18 layer edges give $2^{18}$ layer labelings, and the invariant functions form a subspace of dimension 1024. The helper below applies $T$ without ever writing it down, using two facts:

- with the step edges summed out, a slab reweights each layer edge independently;
- the invariant functions are spanned by products of labels along *even subgraphs* of the layer (every vertex touched an even number of times), because for an abelian group invariants are functions of loop products (glossary 3.5).

Its derivation is in the helper's comments.

In [ ]:
# --- helpers: the exact transfer matrix of the 3D coin model on a W x W torus layer ----------------
# Layer labeling a in {+1,-1}^n (n = 2 W^2 layer edges). Summing the step edges over all their values is
# the same as summing over reframings of the upper layer (Fact 5.1), so on invariant functions
#     T = 2^(W^2) * D^(1/2) K D^(1/2),   K(a,b) = prod_e exp(-beta (1 - a_e b_e)),
# with D(a) = exp(-beta * total cost of the faces inside the layer) (split half and half between slabs).
# K is a tensor power of a 2x2 matrix, diagonal in the "parity" basis chi_S(a) = prod_{e in S} a_e,
# with eigenvalue (1+q)^(n-|S|) (1-q)^|S|, q = exp(-2 beta). chi_S is invariant exactly when S has even
# degree at every vertex. We apply K with a fast Walsh-Hadamard transform (n passes of sums/differences).

def layer_geometry(W):
    edges = [((x, y), mu) for x in range(W) for y in range(W) for mu in (0, 1)]
    eid = {e: i for i, e in enumerate(edges)}
    head = lambda v, mu: ((v[0] + (mu == 0)) % W, (v[1] + (mu == 1)) % W)
    faces = [[eid[((x, y), 0)], eid[(head((x, y), 0), 1)], eid[(head((x, y), 1), 0)], eid[((x, y), 1)]]
             for x in range(W) for y in range(W)]
    at_vertex = {}
    for (v, mu), i in eid.items():
        at_vertex.setdefault(v, []).append(i); at_vertex.setdefault(head(v, mu), []).append(i)
    return len(edges), faces, at_vertex

def walsh(f, n):
    x = f.reshape((2,) * n)
    for ax in range(n):
        a, b = np.take(x, 0, axis=ax), np.take(x, 1, axis=ax)
        x = np.stack([a + b, a - b], axis=ax)
    return x.reshape(-1)

class CoinLayerTransfer:
    """T / 2^(W^2) restricted to invariant functions of a W x W torus layer of the 3D coin model."""
    def __init__(self, W, beta):
        n, faces, at_vertex = layer_geometry(W)
        idx = np.arange(2 ** n, dtype=np.int64)
        bit = lambda i: np.int64(1) << (n - 1 - i)
        self.label = lambda i: 1 - 2 * ((idx & bit(i)) != 0)        # value of layer edge i, per labeling
        S_in = sum(1 - np.prod([self.label(e) for e in f], axis=0) for f in faces)
        self.Dh, self.n, self.faces = np.exp(-beta * S_in / 2), n, faces
        pop = lambda m: np.bitwise_count((idx & m).astype(np.uint64)).astype(np.int64)
        even = np.ones(2 ** n, dtype=bool)
        for es in at_vertex.values():
            even &= pop(np.bitwise_or.reduce([bit(e) for e in es])) % 2 == 0
        q, size = np.exp(-2 * beta), pop(np.int64(-1) & (2 ** n - 1))
        self.diag = even * (1 + q) ** (n - size) * (1 - q) ** size / 2 ** n
    def apply(self, f):
        return self.Dh * walsh(self.diag * walsh(self.Dh * f, self.n), self.n)
    def top(self, k=2):
        op = LinearOperator((2 ** self.n,) * 2, matvec=self.apply, dtype=float)
        vals, vecs = eigsh(op, k=k, which="LA")
        order = np.argsort(vals)[::-1]
        return vals[order], vecs[:, order]

First, a sanity check of the helper. The face product of one face inside layer 0, $A=U_f$, is an invariant. On a board with infinitely many layers, the average of $A(a_0)A(a_t)$ is $\psi_0^{\top}M_A(T/\lambda_0)^tM_A\psi_0$, where $\psi_0$ is the top eigenvector and $M_A$ multiplies by $A$ (chapter 5, section 5.5). Fact 7.4 bounds the correlation by $2\cdot2\cdot4\,\alpha^t/(1-\alpha)$, with $\alpha\le12\tanh\beta$ in 3D.

In [ ]:
beta = 0.05
TL = CoinLayerTransfer(3, beta)
(lam0, _), vecs = TL.top()
psi0 = vecs[:, 0]
A = np.prod([TL.label(e) for e in TL.faces[0]], axis=0).astype(float)       # one face product
v, mean_A = A * psi0, psi0 @ (A * psi0)
alpha = 12 * np.tanh(beta)
print(f"beta = {beta}, alpha <= {alpha:.3f}")
for t in range(1, 3):
    v = TL.apply(v) / lam0
    corr = psi0 @ (A * v) - mean_A ** 2
    print(f"t = {t}: exact correlation {corr:.3e}   Fact 7.4 bound {16 * alpha ** t / (1 - alpha):.3e}")
print(f"tanh(beta)^4 = {np.tanh(beta) ** 4:.3e}")

The exact correlation falls by a factor of about $\tanh(\beta)^4\approx6\times10^{-6}$ per layer. (Beyond $t=2$ it drops below rounding error.) The guaranteed rate is $\alpha=0.6$ per layer, and the guaranteed constant is far above 1 at these distances. The bound is extremely generous, but it is a *proof*, and it holds for every board width at once. The factor $\tanh(\beta)^4$ has a simple meaning: to correlate a face in layer $t$ with one in layer $0$, the model has to pass the information through the four side faces of the cube between them.

> **Recap.** Reweighting by $B$ creates a model that agrees with the original everywhere except on $B$'s edges. Coupling samplers of the two models shows that the disagreement leaks from $S_B$ to $S_A$ only along chains of influences, each costing a factor $\alpha$. So correlations decay like $\alpha^{\text{distance}}$.

**Check yourself**

1. Fact 7.4 has $|S_A|$ but not $|S_B|$ in the bound. Why the asymmetry?

<details><summary>Answer</summary>

Disagreements are *created* on $S_B$, with probability at most 1 per edge. But we bound their arrival at each edge of $S_A$ by a sum over all sources, which is already included in $\rho^*$. Then we add over the edges of $S_A$ that $A$ reads. (Applying the fact with $A$ and $B$ swapped gives the other version; use whichever is smaller.)

</details>

2. Does Fact 7.4 say anything about non-invariant functions, such as a single label?

<details><summary>Answer</summary>

Yes. It holds for all functions, but for a single label it is vacuous: single labels average to zero (chapter 3), and so do their products at different edges. The real content is about invariants such as face products and loop observables.

</details>

## 7.4 From decorrelation to a gap

Chapter 5 showed, by experiment, that the gap is the rate at which layers forget each other. Now we can run that observation backwards as a proof. Decorrelation at rate $\alpha$ per layer forces *every* level to sit at least $\ln(1/\alpha)$ above the lowest.

> **Fact 7.5 (a provable gap).** Take a layered torus grid in $d$ dimensions whose layers are $(d-1)$-dimensional tori of side $W\ge3$. Let $G$ be any compact group and $\varphi$ any face cost of spread $\Phi$. If $\alpha=6(d-1)\tanh(\beta\Phi/2)<1$, then every level of the transfer matrix satisfies $E_n\ge\ln(1/\alpha)$ for $n\ge1$. In particular the gap is at least $\ln(1/\alpha)$, **for every width $W$**.

*Proof.* We write it for finite $G$. For compact $G$, replace sums by integrals; the Perron–Frobenius theorem for matrices is then replaced by its version for positive integral kernels (Jentzsch's theorem).

**Step 1 (the top eigenvector is positive).** Every entry $T(a,b)$ is a sum of positive weights, and $T$ is symmetric (chapter 5). By the Perron–Frobenius theorem, $\lambda_0$ is simple and has an eigenvector $\psi_0$ with every entry positive.

**Step 2 (a test function for each level).** Let $\psi_n$ be a unit eigenvector for $\lambda_n$, $n\ge1$, so $\psi_n\perp\psi_0$. Define the function of a layer labeling $F=\psi_n/\psi_0$; this is legal because $\psi_0>0$. On a board with $N$ layers, the average of $F(a_0)F(a_t)$ is $\operatorname{tr}(M_FT^tM_FT^{N-t})/\operatorname{tr}T^N$. As $N\to\infty$, $T^{N-t}/\lambda_0^{N-t}$ tends to $\psi_0\psi_0^\top$. So the average tends to

$$
\frac{\psi_0^\top M_FT^tM_F\psi_0}{\lambda_0^t}=\frac{\psi_n^\top T^t\psi_n}{\lambda_0^t}=\Big(\frac{\lambda_n}{\lambda_0}\Big)^t,
$$

and $\langle F\rangle\to\psi_0^\top\psi_n=0$.

**Step 3 (apply Fact 7.4).** $F(a_0)$ reads the $(d-1)W^{d-1}$ layer edges of layer 0, and $F(a_t)$ those of layer $t$. A face touches at most two consecutive layers, so the influence distance between layer 0 and layer $t$ is at least $\min(t,N-t)$. Fact 7.4 gives, uniformly in $N$,

$$
\big|\langle F(a_0)F(a_t)\rangle-\langle F\rangle^2\big|\le K\alpha^{\min(t,N-t)},\qquad K=\operatorname{osc}(F)^2\,(d-1)W^{d-1}/(1-\alpha).
$$

**Step 4 (take roots).** Let $N\to\infty$: $(\lambda_n/\lambda_0)^t\le K\alpha^t$ for every $t$. Take $t$-th roots and let $t\to\infty$: $\lambda_n/\lambda_0\le\alpha$. $\square$

Step 4 is where the width disappears. $K$ grows with $W$, but it is a constant in $t$, and the $t$-th root kills it. The *rate* $\alpha$ only knows about one edge and its $6(d-1)$ neighbors.

Now the comparison with exact gaps. For the 2D coin model chapter 5 gave $E_1=-W\ln\tanh\beta$. For the 3D coin model with $3\times3$ layers, the helper computes it.

In [ ]:
betas = np.geomspace(0.004, 1.0, 26)
E1_3d = []
for b in betas:
    (l0, l1), _ = CoinLayerTransfer(3, b).top()
    E1_3d.append(-np.log(l1 / l0))
E1_3d = np.array(E1_3d)
floor = lambda d: np.where(6 * (d - 1) * np.tanh(betas) < 1, -np.log(6 * (d - 1) * np.tanh(betas)), np.nan)
print("beta     exact E1 (3D, W=3)   proven floor (3D)")
for i in [0, 8, 12, 14]:
    print(f"{betas[i]:.4f}   {E1_3d[i]:8.3f}             {floor(3)[i]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(betas, -3 * np.log(np.tanh(betas)), color=viz.MUTED, lw=1.5, label="exact gap, 2D, $W=3$")
ax.plot(betas, E1_3d, "o-", ms=3, color=viz.INK, label="exact gap, 3D, $3\\times3$ layers")
ax.plot(betas, floor(2), color=viz.TEAL, lw=2.5, label="proven floor $\\ln(1/\\alpha)$, 2D (any $W$)")
ax.plot(betas, floor(3), color=viz.CORAL, lw=2.5, label="proven floor $\\ln(1/\\alpha)$, 3D (any $W$)")
ax.axvspan(betas[0], np.arctanh(1 / 12), color=viz.CORAL, alpha=0.08)
ax.set(xscale="log", ylim=(0, 18), xlabel=r"stiffness $\beta$", ylabel="gap $E_1$",
       title="Coin model: true gaps and the floor that Fact 7.5 proves")
ax.legend(fontsize=9); plt.show()

Look at the shaded region, where the 3D guarantee applies: the coral floor stays under the exact gap, as it must. It is far under, by a factor of about 8. The floor is crude but universal. It holds for every width at once and for every compact group with the same spread, and it knew nothing about this particular board beyond "each edge has 12 neighbors". As $\beta$ grows past $\operatorname{artanh}(1/12)\approx0.083$, the floor drops to zero and the theorem says nothing. The true 3D gap keeps going, and becomes small near $\beta\approx0.76$, where the 3D coin model has a jump of its own.

One more observation: at small $\beta$ the exact 3D gap is about $-3\ln\tanh\beta$, the same as the 2D gap with $W=3$. The lightest state is a loop of length 3 wrapping around the layer. That is the "flux loop" of chapter 5, whose level grows with its length.

> **Recap.** Each level $E_n$ is the decay rate of the correlation of a specific layer function, $F=\psi_n/\psi_0$. Fact 7.4 bounds every such correlation by a constant times $\alpha^t$. Taking $t$-th roots removes the constant, so $E_n\ge\ln(1/\alpha)$ for every $n\ge1$ and every width. The exact gaps of the coin model confirm the floor.

**Check yourself**

1. Why do we need $\psi_0$ to be positive everywhere?

<details><summary>Answer</summary>

To divide by it: $F=\psi_n/\psi_0$ must be a genuine function of the layer labeling, with $M_F\psi_0=\psi_n$. Perron–Frobenius guarantees positivity because every entry of $T$ is positive.

</details>

2. For 4D $SU(2)$ with $\varphi=2-\operatorname{tr}$, what does Fact 7.5 prove at $\beta=0.01$?

<details><summary>Answer</summary>

$\Phi=4$, so $\alpha\le18\tanh(0.02)\approx0.36$, and $E_1\ge\ln(1/0.36)\approx1.02$ on every 4D torus grid, whatever its width. The correlation length is at most about 1 grid step.

</details>

3. The 2D gap $-W\ln\tanh\beta$ grows with $W$. Is that consistent with a floor that does not depend on $W$?

<details><summary>Answer</summary>

Yes. A floor is only a lower bound, and $-W\ln\tanh\beta\ge-3\ln\tanh\beta>\ln(1/(6\tanh\beta))$ for all $W\ge3$. The floor cannot grow with $W$ because it is proved one edge at a time, without ever looking at the size of the layer.

</details>

## 7.5 Why this does not prove the Millennium gap

Fact 7.5 is a real theorem about lattice Yang–Mills. With $G=SU(N)$ on a 4D grid it is the small-$\beta$ entry "positive gap at small $\beta$, proved" in chapter 6's status table, re-derived with a computer-science tool. Here is how far it reaches:

| Model (4D) | Spread $\Phi$ | Fact 7.5 applies for our $\beta<$ | physicists' $\beta_{\text{phys}}<$ | Where continuum physics is simulated ($\beta_{\text{phys}}$) |
|---|---|---|---|---|
| coin | 2 | 0.056 | – | – |
| $SU(2)$ | 4 | 0.028 | 0.056 | 2.3 – 3 |
| $SU(3)$ | 4.5 | 0.025 | 0.074 | 5.7 – 7 |

The gap between the last two columns is a factor of 40 to 100. But the true obstruction is not the size of the constants. There are three separate reasons, in increasing order of depth.

**1. Influence grows with stiffness.** At large $\beta$ each edge listens strongly to its neighbors, so $\alpha>1$. This is not only a weakness of the bound: as correlations reach further, the heat-bath sampler of chapter 4 really does slow down, because forgetting in space and forgetting in time go together. Sharper tools from computer science ("spectral independence", block dynamics, "high-dimensional expanders") push the threshold further. But they all certify the same kind of thing: correlations die within a bounded number of grid steps, with constants fixed in advance. Reason 3 shows why that kind of certificate must fail along the continuum path.

**2. The board's geometry is not the issue.** The hint "use expander graphs" points at a real principle, *local-to-global*: a local condition on a sparse graph ($\alpha<1$ on the influence graph) gives a global conclusion. But Fact 7.5 already works on the 4D grid, which is a poor expander: a box of side $\ell$ has $\ell^4$ vertices but only about $\ell^3$ boundary edges. Expansion of the *board* plays no role. What matters is the weakness of each link, and that is exactly what fails at large $\beta$.

**3. The scale problem (the deep one).** The Millennium gap is not "the gap is positive". From chapter 6: as the grid spacing $\varepsilon\to0$, with $\beta(\varepsilon)$ growing like $\tfrac{11}{6\pi^2}\ln(1/\varepsilon)$ for $SU(2)$, one needs

$$
\frac{E_1(\beta(\varepsilon))}{\varepsilon}\ \longrightarrow\ \Delta,\qquad 0<\Delta<\infty .
$$

A bound like $E_1\ge\ln(1/\alpha)$, a constant in grid units, would give $E_1/\varepsilon\to\infty$ if it held along the continuum path: every particle infinitely heavy, and the limit would be trivial. So a proof must show that the gap in grid units **shrinks**, exponentially in $\beta$, at the rate set by asymptotic freedom, and no faster. That is a statement connecting all scales at once. No single-scale argument can deliver it.

**What a proof would need, in our terms.** Group the grid into blocks of side $2^k$. The distribution of suitable block labels is again a model, with a more complicated but still local cost. Asymptotic freedom says its effective stiffness *decreases* as the blocks grow. So the natural plan, Balaban's, is:

- **(UV)** Control the blocking from spacing $\varepsilon$ up to a fixed physical block size $\ell$, uniformly as $\varepsilon\to0$. **This is what the 2026 proof does**, up to a reference scale where the stiffness equals a fixed $\beta_{\text{ref}}$ (see `notes/yang_mills_existence_2026.md`).
- **(Crossover)** Keep blocking until the effective stiffness falls below the threshold of a Fact-7.5-type theorem, while keeping the effective cost local.
- **(IR)** Apply a Fact-7.5-type theorem, generalized to local costs on several faces (the proof goes through verbatim: influence is still a sum over shared terms of $\tanh(\text{spread}/2)$), to the blocked model. Then show that fine-scale observables, given the block labels, are themselves local. The result would be a gap of at least $\ln(1/\alpha_{\text{eff}})/\ell$ in physical units.

The first step is done. The last is this chapter, plus a locality lemma. The middle step is the open problem. The UV methods work by expanding around consistent labelings, which needs large stiffness. The IR method needs stiffness below about 0.03. Between them is the smooth crossover you saw in chapter 6's Creutz plot, where the model is neither nearly consistent nor nearly independent, and no known method controls it. The 2026 proof itself avoids this region. Per the committee's assessment, its large-volume step rests on compactness, flip-positivity and the transfer-matrix representation, and it proves no clustering of correlations. That is why it obtains existence along a subsequence, a mixture of phases, and no gap.

> **Recap.** Fact 7.5 covers only $\beta$ below a few hundredths, far from where the continuum lives. Beyond that, influence is too strong. More fundamentally, the Millennium gap needs the grid-unit gap to shrink at exactly the asymptotic-freedom rate, which is a multi-scale statement. The missing piece is control of the crossover between the near-consistent regime (handled by the 2026 existence proof) and the near-independent regime (handled by this chapter).

**Check yourself**

1. Suppose someone proves $E_1(\beta)\ge0.1$ for all $\beta$ in 4D $SU(2)$. Would that solve the Millennium gap?

<details><summary>Answer</summary>

No, it would refute the expected continuum limit. Along the continuum path the gap in physical units would be $E_1/\varepsilon\ge0.1/\varepsilon\to\infty$, so the limiting theory would have no particles at all. Physicists expect $E_1(\beta)\to0$ like $e^{-6\pi^2\beta/11}$.

</details>

2. What lower bound on $E_1(\beta)$ in grid units, for large $\beta$, would a solution need to prove (for $SU(2)$, leading order)?

<details><summary>Answer</summary>

$E_1(\beta)\ge c\,e^{-6\pi^2\beta/11}$ for some $c>0$ and all large $\beta$, along with the existence of the limit. Then $E_1/\varepsilon$ stays bounded below. (An upper bound of the same form, needed for a non-trivial limit, is expected from asymptotic freedom.)

</details>

## Chapter recap

- The **influence** $C(e,e')$ is the most the heat-bath distribution at $e$ can move when the label at $e'$ changes. The **influence number** $\alpha$ is the largest total influence on one edge. On a grid, $\alpha\le6(d-1)\tanh(\beta\Phi/2)$ (Facts 7.1, 7.2).
- A **coupling** runs two random processes on shared randomness. With best couplings, two heat-bath samplers disagree at any edge with probability at most $\alpha^t$ after $t$ sweeps (Fact 7.3). This is the path-coupling argument for rapid mixing.
- Reweighting by $B$ changes the model only on $B$'s edges. Coupling the two models gives correlations at most $\operatorname{osc}(A)\operatorname{osc}(B)|S_A|\alpha^D/(1-\alpha)$ (Fact 7.4).
- Testing with $F=\psi_n/\psi_0$ turns decorrelation into a floor under every level: $E_n\ge\ln(1/\alpha)$ for every width, every compact group, every dimension (Fact 7.5). Checked against exact 2D and 3D coin-model gaps.
- For 4D $SU(2)$ the theorem applies for $\beta<0.028$. The continuum needs $\beta\to\infty$ with a gap that **shrinks** at the asymptotic-freedom rate, a multi-scale statement no single-scale argument can give.
- The missing step is the crossover between the regime the 2026 existence proof controls and the regime this chapter controls.

## Dictionary

| Our term | Other names |
|---|---|
| distance between distributions $d(p,q)$ | total variation distance |
| best coupling | maximal (optimal) coupling |
| influence $C(e,e')$ | Dobrushin interdependence matrix |
| $\alpha<1$ | Dobrushin's uniqueness condition; in physics, a "high-temperature" or "strong-coupling" condition |
| two samplers on the same random numbers | synchronous coupling; path coupling (Bubley–Dyer) |
| sampler forgets geometrically | rapid mixing of Glauber dynamics |
| correlations decay with influence distance | spatial mixing, correlation decay, uniqueness of the Gibbs measure |
| Fact 7.5 for $SU(N)$ | mass gap at strong coupling (Osterwalder–Seiler 1978, by cluster expansion) |
| blocking the grid | renormalization group (block-spin transformation); Balaban's program |
| the crossover | the intermediate-coupling regime; the gap between perturbative and strong-coupling control |

## What's next

This chapter ends the elementary road. Two directions continue it. On the computer-science side: can spectral independence or block (multi-scale) dynamics give a gap that shrinks at a *controlled* rate for the 3D coin model, where the answer is known from its exact relation to the 3D Ising model? On the physics side: the 2026 proof's blocking machinery, read in our terms, is the natural place to look for the crossover step. Read the notes in `notes/yang_mills_existence_2026.md` with this chapter's three-step plan in mind. The research memo `notes/crossover/README.md` takes the middle step further: it shows that the simplest blocking scheme runs on the curvature of $SU(2)$ (and stalls for the circle), measures how wide the crossover really is, and lists what a certificate tight enough to cross it would need.